# 📘 Topic-Conditioned Narrative Shift Detection using TCL

## Complete Implementation Pipeline

**Framework:** Temporal Contrastive Learning (TCL)  
**Architecture:** Transformer-based encoder with soft topic conditioning  
**Objective:** Detect narrative shifts across 5 topics (War, Health, Economics, Technology, Climate)

---

### Pipeline Stages:
1. Data Loading & Parsing
2. Topic-Specific Daily Aggregation
3. Temporal Gap Modeling
4. Window Construction
5. Dataset Merging
6. TCL Model Training
7. Macro Drift Detection
8. Micro Pivot Detection
9. Results Visualization

In [ ]:
# Cell 1: Environment Setup and Imports

import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm
import warnings
import json
import pickle
from pathlib import Path

warnings.filterwarnings('ignore')

# Environment Detection
IS_KAGGLE = os.path.exists('/kaggle/input')

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA Device: {torch.cuda.get_device_name(0)}")

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Cell 2: Configuration

class Config:
    """Global configuration for the pipeline"""
    
    # Paths
    if IS_KAGGLE:
        DATA_PATH = '/kaggle/input/datasets/prateek1005/data-embedding'
        OUTPUT_PATH = '/kaggle/working'
    else:
        DATA_PATH = 'Processed_Data/Topic_Wise_w3'
        OUTPUT_PATH = 'Narrative_Shift_Results'
    
    # Topics
    TOPICS = ['War', 'Health', 'Economics', 'Technology', 'Climate']
    TOPIC_FILES = {
        'War': 'War.csv',
        'Health': 'Health.csv',
        'Economics': 'Economics.csv',
        'Technology': 'Technology.csv',
        'Climate': 'Climate.csv'
    }
    
    # Data Processing
    EMBEDDING_DIM = 768
    TOPIC_DIM = 5
    TIME_DIM = 1
    FINAL_DIM = EMBEDDING_DIM + TIME_DIM + TOPIC_DIM  # 774
    TOPIC_THRESHOLD = 0.1  # Minimum topic probability for inclusion
    
    # Window Configuration
    WINDOW_SIZE = 30  # days
    WINDOW_STRIDE = 1  # days
    
    # Model Architecture
    HIDDEN_DIM = 256
    PROJECTION_DIM = 128
    NUM_LAYERS = 2
    NUM_HEADS = 4
    DROPOUT = 0.1
    
    # Training
    BATCH_SIZE = 64
    EPOCHS = 50
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    TEMPERATURE = 0.07
    
    # Drift Detection
    ZSCORE_THRESHOLD = 2.0
    PERCENTILE_THRESHOLD = 95  # Top 5%
    
    # Output
    SAVE_CHECKPOINTS = True
    CHECKPOINT_FREQ = 10  # epochs

config = Config()

# Create output directory
os.makedirs(config.OUTPUT_PATH, exist_ok=True)

print("Configuration loaded:")
print(f"  Data Path: {config.DATA_PATH}")
print(f"  Output Path: {config.OUTPUT_PATH}")
print(f"  Topics: {config.TOPICS}")
print(f"  Window Size: {config.WINDOW_SIZE} days")
print(f"  Final Vector Dim: {config.FINAL_DIM}")

---
## 🔵 STAGE 1: Data Loading and Parsing

In [ ]:
# Cell 3: Data Loading Functions

def parse_embedding(emb_str):
    """
    Parse embedding string to numpy array.
    Handles both comma-separated and space-separated formats.
    """
    if isinstance(emb_str, np.ndarray):
        return emb_str.astype(np.float32)
    
    if isinstance(emb_str, str):
        # Remove quotes and brackets
        emb_str = emb_str.strip('"\' []')
        
        # Try comma-separated first
        if ',' in emb_str:
            values = [float(x.strip()) for x in emb_str.split(',') if x.strip()]
        else:
            # Fallback to space-separated
            values = [float(x) for x in emb_str.split() if x]
        
        return np.array(values, dtype=np.float32)
    
    raise ValueError(f"Unsupported embedding format: {type(emb_str)}")


def load_topic_data(topic_name):
    """
    Load and parse data for a specific topic.
    
    Returns:
        DataFrame with columns: date, w3_embedding, topic_probs, main_sentence
    """
    filepath = os.path.join(config.DATA_PATH, config.TOPIC_FILES[topic_name])
    
    print(f"\nLoading {topic_name} data from: {filepath}")
    
    if not os.path.exists(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    
    # Load CSV
    df = pd.read_csv(filepath)
    print(f"  Loaded {len(df)} rows")
    
    # Parse dates (flexible format)
    df['date'] = pd.to_datetime(df['date'], format='mixed', errors='coerce')
    
    # Drop rows with invalid dates
    invalid_dates = df['date'].isna().sum()
    if invalid_dates > 0:
        print(f"  Warning: Dropping {invalid_dates} rows with invalid dates")
        df = df.dropna(subset=['date'])
    
    # Sort by date
    df = df.sort_values('date').reset_index(drop=True)
    
    # Parse embeddings
    print("  Parsing embeddings...")
    df['w3_embedding'] = df['w3_embedding'].apply(parse_embedding)
    
    # Validate embedding dimensions
    invalid_emb = df['w3_embedding'].apply(lambda x: len(x) != config.EMBEDDING_DIM)
    if invalid_emb.any():
        print(f"  Warning: Dropping {invalid_emb.sum()} rows with invalid embedding dimensions")
        df = df[~invalid_emb].reset_index(drop=True)
    
    # Extract topic probability vector
    topic_cols = config.TOPICS
    df['topic_probs'] = df[topic_cols].apply(
        lambda row: np.array(row.values, dtype=np.float32), axis=1
    )
    
    print(f"  Final dataset: {len(df)} rows")
    print(f"  Date range: {df['date'].min()} to {df['date'].max()}")
    
    return df[['date', 'w3_embedding', 'topic_probs', 'main_sentence']]

---
## 🔵 STAGE 2: Topic-Specific Daily Aggregation

In [ ]:
# Cell 4: Daily Aggregation Function

def aggregate_daily_weighted(df, topic_idx):
    """
    Aggregate sentence embeddings daily using weighted mean based on topic probability.
    
    Args:
        df: DataFrame with date, w3_embedding, topic_probs
        topic_idx: Index of the topic (0-4)
    
    Returns:
        List of dictionaries with daily aggregated data
    """
    print(f"\n  Aggregating daily data for topic index {topic_idx}...")
    
    # Extract date only (no time)
    df['date_only'] = df['date'].dt.date
    
    daily_data = []
    
    for date, group in df.groupby('date_only'):
        # Extract weights (topic probability for this topic)
        weights = group['topic_probs'].apply(lambda x: x[topic_idx]).values
        
        # Skip if total weight is zero
        if weights.sum() == 0:
            continue
        
        # Weighted semantic mean
        embeddings = np.stack(group['w3_embedding'].values)
        semantic_vector = np.average(embeddings, axis=0, weights=weights)
        
        # Weighted topic mean
        topic_probs = np.stack(group['topic_probs'].values)
        topic_vector = np.average(topic_probs, axis=0, weights=weights)
        
        # Apply topic presence filter
        if topic_vector[topic_idx] < config.TOPIC_THRESHOLD:
            continue
        
        daily_data.append({
            'date': pd.Timestamp(date),
            'semantic_vector': semantic_vector.astype(np.float32),
            'topic_vector': topic_vector.astype(np.float32),
            'num_sentences': len(group),
            'total_weight': weights.sum()
        })
    
    print(f"    Valid days after filtering: {len(daily_data)}")
    
    return daily_data

---
## 🔵 STAGE 3: Temporal Gap Modeling

In [ ]:
# Cell 5: Add Time Gap Features

def add_time_gap_features(daily_data):
    """
    Add time gap feature (log(1 + delta_days)) to daily vectors.
    
    Final vector: [semantic_vector (768), tau (1), topic_vector (5)] = 774
    """
    print(f"\n  Adding time gap features...")
    
    # Sort by date
    daily_data = sorted(daily_data, key=lambda x: x['date'])
    
    enhanced_data = []
    
    for i, entry in enumerate(daily_data):
        # Calculate time gap
        if i == 0:
            tau = 0.0
        else:
            delta_days = (entry['date'] - daily_data[i-1]['date']).days
            tau = np.log(1 + delta_days)
        
        # Construct final vector: [semantic (768), tau (1), topic (5)]
        final_vector = np.concatenate([
            entry['semantic_vector'],  # 768
            np.array([tau], dtype=np.float32),  # 1
            entry['topic_vector']  # 5
        ])
        
        enhanced_data.append({
            'date': entry['date'],
            'vector': final_vector,  # 774 dimensions
            'time_gap': tau,
            'num_sentences': entry['num_sentences']
        })
    
    print(f"    Enhanced {len(enhanced_data)} daily vectors")
    print(f"    Final vector dimension: {enhanced_data[0]['vector'].shape[0]}")
    
    return enhanced_data

---
## 🔵 STAGE 4: Window Construction

In [ ]:
# Cell 6: Window Construction Function

def create_sliding_windows(daily_data, topic_name, topic_idx):
    """
    Create sliding windows from daily data.
    
    Args:
        daily_data: List of daily vectors
        topic_name: Name of the topic
        topic_idx: Index of the topic (0-4)
    
    Returns:
        List of window dictionaries
    """
    print(f"\n  Creating sliding windows for {topic_name}...")
    
    windows = []
    n_days = len(daily_data)
    
    for i in range(n_days - config.WINDOW_SIZE + 1):
        # Extract window
        window_entries = daily_data[i:i + config.WINDOW_SIZE]
        
        # Stack vectors into tensor (30, 774)
        window_tensor = np.stack([entry['vector'] for entry in window_entries])
        
        windows.append({
            'tensor': window_tensor.astype(np.float32),
            'topic_id': topic_idx,
            'topic_name': topic_name,
            'start_date': window_entries[0]['date'],
            'end_date': window_entries[-1]['date'],
            'window_idx': i
        })
    
    print(f"    Created {len(windows)} windows")
    
    return windows

In [ ]:
# Cell 7: Process All Topics and Create Windows

def process_all_topics():
    """
    Process all topics and create windows.
    
    Returns:
        all_windows: Combined list of all windows
        topic_windows: Dictionary mapping topic_name -> list of windows
    """
    all_windows = []
    topic_windows = {}
    
    for topic_idx, topic_name in enumerate(config.TOPICS):
        print(f"\n{'='*60}")
        print(f"Processing Topic: {topic_name} (index {topic_idx})")
        print(f"{'='*60}")
        
        # Load data
        df = load_topic_data(topic_name)
        
        # Daily aggregation
        daily_data = aggregate_daily_weighted(df, topic_idx)
        
        # Add time gap features
        enhanced_data = add_time_gap_features(daily_data)
        
        # Create windows
        windows = create_sliding_windows(enhanced_data, topic_name, topic_idx)
        
        # Store
        topic_windows[topic_name] = windows
        all_windows.extend(windows)
        
        print(f"\n  ✓ {topic_name}: {len(windows)} windows created")
    
    print(f"\n{'='*60}")
    print(f"Total windows across all topics: {len(all_windows)}")
    print(f"{'='*60}")
    
    return all_windows, topic_windows


# Process all topics
print("Processing all topics...\n")
all_windows, topic_windows = process_all_topics()

# Summary
print("\n\nSummary by Topic:")
for topic_name, windows in topic_windows.items():
    print(f"  {topic_name:15s}: {len(windows):6d} windows")

---
## 🔵 STAGE 5: Dataset Merging and Preparation

In [ ]:
# Cell 8: PyTorch Dataset Class

class TemporalWindowDataset(Dataset):
    """
    PyTorch Dataset for temporal windows.
    Supports contrastive learning with consecutive window pairs.
    """
    
    def __init__(self, windows, shuffle=True):
        """
        Args:
            windows: List of window dictionaries
            shuffle: Whether to shuffle windows
        """
        self.windows = windows.copy()
        
        if shuffle:
            np.random.shuffle(self.windows)
        
        # Group by topic for consecutive pair sampling
        self.topic_grouped = {}
        for w in self.windows:
            topic = w['topic_name']
            if topic not in self.topic_grouped:
                self.topic_grouped[topic] = []
            self.topic_grouped[topic].append(w)
        
        # Sort each topic group by window_idx for consecutive pairing
        for topic in self.topic_grouped:
            self.topic_grouped[topic] = sorted(
                self.topic_grouped[topic], 
                key=lambda x: x['window_idx']
            )
    
    def __len__(self):
        return len(self.windows)
    
    def __getitem__(self, idx):
        window = self.windows[idx]
        tensor = torch.from_numpy(window['tensor'])  # (30, 774)
        topic_id = window['topic_id']
        
        return tensor, topic_id
    
    def get_consecutive_pair_batch(self, batch_size):
        """
        Sample consecutive window pairs for contrastive learning.
        Returns: (anchors, positives, negatives)
        """
        anchors = []
        positives = []
        
        # Sample pairs from each topic
        samples_per_topic = batch_size // len(config.TOPICS)
        
        for topic in config.TOPICS:
            topic_windows = self.topic_grouped[topic]
            
            # Sample random starting indices
            max_idx = len(topic_windows) - 1
            if max_idx <= 0:
                continue
            
            indices = np.random.randint(0, max_idx, size=samples_per_topic)
            
            for i in indices:
                anchors.append(torch.from_numpy(topic_windows[i]['tensor']))
                positives.append(torch.from_numpy(topic_windows[i+1]['tensor']))
        
        # Stack into batches
        anchors = torch.stack(anchors)
        positives = torch.stack(positives)
        
        return anchors, positives


# Create dataset
print("Creating PyTorch dataset...")
dataset = TemporalWindowDataset(all_windows, shuffle=True)
print(f"  Dataset size: {len(dataset)}")

---
## 🔵 STAGE 6: TCL Model Architecture

In [ ]:
# Cell 9: Temporal Encoder Model

class TemporalEncoder(nn.Module):
    """
    Transformer-based temporal encoder for narrative shift detection.
    
    Architecture:
    1. Input projection (774 -> 256)
    2. Positional encoding
    3. Transformer encoder layers
    4. Mean pooling
    5. Projection head (256 -> 128)
    6. L2 normalization
    """
    
    def __init__(self, config):
        super().__init__()
        
        # Input projection
        self.input_proj = nn.Linear(config.FINAL_DIM, config.HIDDEN_DIM)
        
        # Learnable positional encoding
        self.pos_encoding = nn.Parameter(
            torch.randn(1, config.WINDOW_SIZE, config.HIDDEN_DIM) * 0.02
        )
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.HIDDEN_DIM,
            nhead=config.NUM_HEADS,
            dim_feedforward=config.HIDDEN_DIM * 4,
            dropout=config.DROPOUT,
            activation='relu',
            batch_first=True
        )
        
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=config.NUM_LAYERS
        )
        
        # Projection head
        self.projection_head = nn.Sequential(
            nn.Linear(config.HIDDEN_DIM, config.HIDDEN_DIM),
            nn.ReLU(),
            nn.Linear(config.HIDDEN_DIM, config.PROJECTION_DIM)
        )
        
        self.config = config
    
    def forward(self, x, return_features=False):
        """
        Args:
            x: (batch, window_size, final_dim) = (B, 30, 774)
            return_features: If True, return intermediate features
        
        Returns:
            z: (batch, projection_dim) = (B, 128) - L2 normalized
        """
        # Input projection
        x = self.input_proj(x)  # (B, 30, 256)
        
        # Add positional encoding
        x = x + self.pos_encoding  # (B, 30, 256)
        
        # Transformer encoding
        x = self.transformer(x)  # (B, 30, 256)
        
        # Mean pooling over time
        features = x.mean(dim=1)  # (B, 256)
        
        # Projection head
        z = self.projection_head(features)  # (B, 128)
        
        # L2 normalization
        z = F.normalize(z, p=2, dim=1)
        
        if return_features:
            return z, features
        return z


# Build model
print("Building model...")
model = TemporalEncoder(config).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# Cell 10: NT-Xent Contrastive Loss

class NTXentLoss(nn.Module):
    """
    Normalized Temperature-scaled Cross Entropy Loss (NT-Xent).
    Used for contrastive learning.
    """
    
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature
    
    def forward(self, z_i, z_j):
        """
        Args:
            z_i: Anchor embeddings (B, D)
            z_j: Positive embeddings (B, D)
        
        Returns:
            loss: Scalar contrastive loss
        """
        batch_size = z_i.shape[0]
        
        # Concatenate anchors and positives
        z = torch.cat([z_i, z_j], dim=0)  # (2B, D)
        
        # Compute similarity matrix
        sim_matrix = torch.mm(z, z.t()) / self.temperature  # (2B, 2B)
        
        # Create mask for positive pairs
        # Positive pairs: (i, i+B) and (i+B, i)
        mask = torch.eye(2 * batch_size, dtype=torch.bool, device=z.device)
        
        # Remove self-similarities
        sim_matrix = sim_matrix.masked_fill(mask, -9e15)
        
        # Positive similarities
        pos_sim = torch.cat([
            torch.diag(sim_matrix, batch_size),
            torch.diag(sim_matrix, -batch_size)
        ], dim=0)  # (2B,)
        
        # Compute loss using log-sum-exp trick
        loss = -pos_sim + torch.logsumexp(sim_matrix, dim=1)
        loss = loss.mean()
        
        return loss

---
## 🔵 STAGE 7: Training Loop

In [ ]:
# Cell 11: Training Function

def train_tcl_model(model, dataset, config):
    """
    Train the TCL model using contrastive learning.
    """
    print("\n" + "="*60)
    print("Starting Training")
    print("="*60)
    
    # Optimizer
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY
    )
    
    # Loss function
    criterion = NTXentLoss(temperature=config.TEMPERATURE)
    
    # Training history
    history = {'epoch': [], 'loss': []}
    
    # Training loop
    model.train()
    
    for epoch in range(config.EPOCHS):
        epoch_losses = []
        
        # Number of batches per epoch
        n_batches = max(len(dataset) // config.BATCH_SIZE, 10)
        
        progress_bar = tqdm(range(n_batches), desc=f"Epoch {epoch+1}/{config.EPOCHS}")
        
        for _ in progress_bar:
            # Sample consecutive pairs
            anchors, positives = dataset.get_consecutive_pair_batch(config.BATCH_SIZE)
            anchors = anchors.to(device)
            positives = positives.to(device)
            
            # Forward pass
            z_anchor = model(anchors)
            z_positive = model(positives)
            
            # Compute loss
            loss = criterion(z_anchor, z_positive)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            # Record loss
            epoch_losses.append(loss.item())
            progress_bar.set_postfix({'loss': f"{loss.item():.4f}"})
        
        # Epoch statistics
        avg_loss = np.mean(epoch_losses)
        history['epoch'].append(epoch + 1)
        history['loss'].append(avg_loss)
        
        print(f"  Epoch {epoch+1}: Loss = {avg_loss:.4f}")
        
        # Save checkpoint
        if config.SAVE_CHECKPOINTS and (epoch + 1) % config.CHECKPOINT_FREQ == 0:
            checkpoint_path = os.path.join(
                config.OUTPUT_PATH,
                f"checkpoint_epoch_{epoch+1}.pt"
            )
            torch.save({
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss,
            }, checkpoint_path)
            print(f"    Checkpoint saved: {checkpoint_path}")
    
    print("\n" + "="*60)
    print("Training Complete")
    print("="*60)
    
    return history


# Train model
print("Starting model training...")
training_history = train_tcl_model(model, dataset, config)

# Plot training curve
plt.figure(figsize=(10, 5))
plt.plot(training_history['epoch'], training_history['loss'], marker='o')
plt.xlabel('Epoch')
plt.ylabel('NT-Xent Loss')
plt.title('Training Loss Curve')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(config.OUTPUT_PATH, 'training_curve.png'), dpi=150)
plt.show()

# Save final model
final_model_path = os.path.join(config.OUTPUT_PATH, 'tcl_model_final.pt')
torch.save(model.state_dict(), final_model_path)
print(f"\nFinal model saved: {final_model_path}")

---
## 🔵 STAGE 8: Macro Drift Detection

In [ ]:
# Cell 12: Drift Score Computation

def compute_drift_scores(model, windows, device):
    """
    Compute drift scores for a sequence of windows.
    
    Args:
        model: Trained encoder model
        windows: List of window dictionaries (sorted chronologically)
        device: torch device
    
    Returns:
        drift_data: List of dictionaries with drift information
    """
    model.eval()
    
    embeddings = []
    
    # Encode all windows
    with torch.no_grad():
        for window in tqdm(windows, desc="Encoding windows"):
            tensor = torch.from_numpy(window['tensor']).unsqueeze(0).to(device)
            z = model(tensor)
            embeddings.append(z.cpu().numpy()[0])
    
    embeddings = np.array(embeddings)  # (N, 128)
    
    # Compute cosine distances
    drift_scores = []
    
    for i in range(1, len(embeddings)):
        # Cosine distance = 1 - cosine similarity
        cos_sim = np.dot(embeddings[i], embeddings[i-1])
        drift = 1 - cos_sim
        drift_scores.append(drift)
    
    drift_scores = np.array(drift_scores)
    
    # Standardize (z-scores)
    mean_drift = drift_scores.mean()
    std_drift = drift_scores.std()
    z_scores = (drift_scores - mean_drift) / (std_drift + 1e-8)
    
    # Build drift data
    drift_data = []
    
    for i, (drift, zscore) in enumerate(zip(drift_scores, z_scores)):
        drift_data.append({
            'window_idx': i + 1,
            'date': windows[i+1]['start_date'],
            'drift_score': drift,
            'z_score': zscore,
            'prev_date': windows[i]['start_date']
        })
    
    return drift_data, embeddings


def detect_shifts(drift_data, config):
    """
    Detect significant narrative shifts.
    
    Returns:
        List of shift events
    """
    shifts = []
    
    # Get z-scores
    z_scores = np.array([d['z_score'] for d in drift_data])
    
    # Threshold 1: Z-score > 2
    zscore_threshold_mask = z_scores > config.ZSCORE_THRESHOLD
    
    # Threshold 2: Top 5%
    percentile_threshold = np.percentile(z_scores, config.PERCENTILE_THRESHOLD)
    percentile_mask = z_scores > percentile_threshold
    
    # Combine (OR operation)
    shift_mask = zscore_threshold_mask | percentile_mask
    
    # Extract shifts
    for i, is_shift in enumerate(shift_mask):
        if is_shift:
            shifts.append(drift_data[i])
    
    return shifts


# Compute drift for all topics
print("\nComputing drift scores for all topics...\n")

topic_drift_data = {}
topic_embeddings = {}
topic_shifts = {}

for topic_name in config.TOPICS:
    print(f"\n{'='*60}")
    print(f"Processing: {topic_name}")
    print(f"{'='*60}")
    
    # Get windows for this topic (sorted)
    windows = sorted(topic_windows[topic_name], key=lambda x: x['window_idx'])
    
    # Compute drift
    drift_data, embeddings = compute_drift_scores(model, windows, device)
    
    # Detect shifts
    shifts = detect_shifts(drift_data, config)
    
    # Store
    topic_drift_data[topic_name] = drift_data
    topic_embeddings[topic_name] = embeddings
    topic_shifts[topic_name] = shifts
    
    print(f"\n  Drift scores computed: {len(drift_data)}")
    print(f"  Shifts detected: {len(shifts)}")
    
    if len(shifts) > 0:
        print(f"\n  Top 5 shifts:")
        top_shifts = sorted(shifts, key=lambda x: x['z_score'], reverse=True)[:5]
        for shift in top_shifts:
            print(f"    {shift['date'].date()}: Z-score = {shift['z_score']:.2f}, Drift = {shift['drift_score']:.4f}")

print("\n" + "="*60)
print("Drift detection complete")
print("="*60)

In [ ]:
# Cell 13: Visualization - Drift Timeline

def plot_drift_timeline(drift_data, shifts, topic_name, output_path):
    """
    Plot drift timeline with detected shifts.
    """
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
    
    dates = [d['date'] for d in drift_data]
    drift_scores = [d['drift_score'] for d in drift_data]
    z_scores = [d['z_score'] for d in drift_data]
    
    # Plot 1: Drift scores
    ax1.plot(dates, drift_scores, alpha=0.7, linewidth=1.5, color='steelblue')
    ax1.set_ylabel('Drift Score', fontsize=12)
    ax1.set_title(f'{topic_name} - Narrative Drift Timeline', fontsize=14, fontweight='bold')
    ax1.grid(True, alpha=0.3)
    
    # Mark shifts
    if len(shifts) > 0:
        shift_dates = [s['date'] for s in shifts]
        shift_scores = [s['drift_score'] for s in shifts]
        ax1.scatter(shift_dates, shift_scores, color='red', s=100, 
                   marker='o', alpha=0.7, label=f'Shifts (n={len(shifts)})', zorder=5)
        ax1.legend()
    
    # Plot 2: Z-scores
    ax2.plot(dates, z_scores, alpha=0.7, linewidth=1.5, color='forestgreen')
    ax2.axhline(y=2, color='red', linestyle='--', alpha=0.5, label='Z-score threshold (2.0)')
    ax2.axhline(y=0, color='black', linestyle='-', alpha=0.3, linewidth=0.5)
    ax2.set_ylabel('Z-Score', fontsize=12)
    ax2.set_xlabel('Date', fontsize=12)
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    
    plt.tight_layout()
    
    # Save
    filepath = os.path.join(output_path, f'drift_timeline_{topic_name}.png')
    plt.savefig(filepath, dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"  Saved: {filepath}")


# Plot for all topics
print("\nGenerating drift timeline visualizations...\n")

for topic_name in config.TOPICS:
    print(f"Plotting {topic_name}...")
    plot_drift_timeline(
        topic_drift_data[topic_name],
        topic_shifts[topic_name],
        topic_name,
        config.OUTPUT_PATH
    )

---
## 🔵 STAGE 9: Micro Pivot Detection

In [ ]:
# Cell 14: Sentence-Level Pivot Detection

def find_pivot_sentence(topic_name, shift_date, topic_idx, window_days=7):
    """
    Find the pivot sentence responsible for a narrative shift.
    
    Args:
        topic_name: Name of topic
        shift_date: Date of detected shift
        topic_idx: Topic index
        window_days: Days before/after shift to search
    
    Returns:
        Dictionary with pivot information
    """
    # Load original data
    df = load_topic_data(topic_name)
    
    # Filter to date range
    start_date = shift_date - timedelta(days=window_days)
    end_date = shift_date + timedelta(days=window_days)
    
    df_region = df[(df['date'] >= start_date) & (df['date'] <= end_date)].copy()
    
    if len(df_region) < 2:
        return None
    
    # Sort chronologically
    df_region = df_region.sort_values('date').reset_index(drop=True)
    
    # Compute sentence-level drift
    sentence_drifts = []
    
    for i in range(1, len(df_region)):
        # Cosine distance
        emb_prev = df_region.iloc[i-1]['w3_embedding']
        emb_curr = df_region.iloc[i]['w3_embedding']
        
        cos_sim = np.dot(emb_prev, emb_curr) / (np.linalg.norm(emb_prev) * np.linalg.norm(emb_curr) + 1e-8)
        drift = 1 - cos_sim
        
        # Time adjustment
        time_gap = (df_region.iloc[i]['date'] - df_region.iloc[i-1]['date']).total_seconds() / 86400  # days
        adjusted_drift = drift / (np.log(1 + time_gap) + 1e-8)
        
        sentence_drifts.append({
            'idx': i,
            'date': df_region.iloc[i]['date'],
            'prev_date': df_region.iloc[i-1]['date'],
            'drift': drift,
            'adjusted_drift': adjusted_drift,
            'time_gap': time_gap,
            'sentence': df_region.iloc[i]['main_sentence'],
            'prev_sentence': df_region.iloc[i-1]['main_sentence']
        })
    
    # Find maximum adjusted drift
    pivot = max(sentence_drifts, key=lambda x: x['adjusted_drift'])
    
    return pivot


# Find pivots for top shifts
print("\nFinding pivot sentences for top shifts...\n")

pivot_results = {}

for topic_idx, topic_name in enumerate(config.TOPICS):
    print(f"\n{'='*60}")
    print(f"Topic: {topic_name}")
    print(f"{'='*60}")
    
    shifts = topic_shifts[topic_name]
    
    if len(shifts) == 0:
        print("  No shifts detected.")
        continue
    
    # Get top 3 shifts
    top_shifts = sorted(shifts, key=lambda x: x['z_score'], reverse=True)[:3]
    
    topic_pivots = []
    
    for rank, shift in enumerate(top_shifts, 1):
        print(f"\n  Shift #{rank}: {shift['date'].date()} (Z-score: {shift['z_score']:.2f})")
        
        pivot = find_pivot_sentence(topic_name, shift['date'], topic_idx)
        
        if pivot:
            print(f"    Pivot found on {pivot['date'].date()}")
            print(f"    Adjusted drift: {pivot['adjusted_drift']:.4f}")
            print(f"    Time gap: {pivot['time_gap']:.2f} days")
            print(f"\n    Previous: {pivot['prev_sentence'][:150]}...")
            print(f"    Current:  {pivot['sentence'][:150]}...")
            
            topic_pivots.append({
                'shift_date': shift['date'],
                'shift_zscore': shift['z_score'],
                'pivot_date': pivot['date'],
                'adjusted_drift': pivot['adjusted_drift'],
                'prev_sentence': pivot['prev_sentence'],
                'curr_sentence': pivot['sentence']
            })
        else:
            print("    No pivot found.")
    
    pivot_results[topic_name] = topic_pivots

print("\n" + "="*60)
print("Pivot detection complete")
print("="*60)

---
## 🔵 Final: Save Results

In [ ]:
# Cell 15: Save All Results

print("\nSaving results...\n")

# 1. Save drift data (CSV)
for topic_name in config.TOPICS:
    df_drift = pd.DataFrame(topic_drift_data[topic_name])
    filepath = os.path.join(config.OUTPUT_PATH, f'drift_scores_{topic_name}.csv')
    df_drift.to_csv(filepath, index=False)
    print(f"Saved: {filepath}")

# 2. Save shifts (JSON)
shifts_serializable = {}
for topic_name, shifts in topic_shifts.items():
    shifts_serializable[topic_name] = [
        {
            'date': str(s['date']),
            'z_score': float(s['z_score']),
            'drift_score': float(s['drift_score'])
        }
        for s in shifts
    ]

shifts_filepath = os.path.join(config.OUTPUT_PATH, 'detected_shifts.json')
with open(shifts_filepath, 'w') as f:
    json.dump(shifts_serializable, f, indent=2)
print(f"Saved: {shifts_filepath}")

# 3. Save pivot results (JSON)
pivots_serializable = {}
for topic_name, pivots in pivot_results.items():
    pivots_serializable[topic_name] = [
        {
            'shift_date': str(p['shift_date']),
            'shift_zscore': float(p['shift_zscore']),
            'pivot_date': str(p['pivot_date']),
            'adjusted_drift': float(p['adjusted_drift']),
            'prev_sentence': p['prev_sentence'],
            'curr_sentence': p['curr_sentence']
        }
        for p in pivots
    ]

pivots_filepath = os.path.join(config.OUTPUT_PATH, 'pivot_sentences.json')
with open(pivots_filepath, 'w') as f:
    json.dump(pivots_serializable, f, indent=2)
print(f"Saved: {pivots_filepath}")

# 4. Save embeddings (pickle)
embeddings_filepath = os.path.join(config.OUTPUT_PATH, 'topic_embeddings.pkl')
with open(embeddings_filepath, 'wb') as f:
    pickle.dump(topic_embeddings, f)
print(f"Saved: {embeddings_filepath}")

# 5. Save configuration
config_dict = {
    'topics': config.TOPICS,
    'window_size': config.WINDOW_SIZE,
    'embedding_dim': config.EMBEDDING_DIM,
    'hidden_dim': config.HIDDEN_DIM,
    'projection_dim': config.PROJECTION_DIM,
    'num_layers': config.NUM_LAYERS,
    'num_heads': config.NUM_HEADS,
    'batch_size': config.BATCH_SIZE,
    'epochs': config.EPOCHS,
    'learning_rate': config.LEARNING_RATE,
    'temperature': config.TEMPERATURE,
    'zscore_threshold': config.ZSCORE_THRESHOLD,
    'percentile_threshold': config.PERCENTILE_THRESHOLD
}

config_filepath = os.path.join(config.OUTPUT_PATH, 'config.json')
with open(config_filepath, 'w') as f:
    json.dump(config_dict, f, indent=2)
print(f"Saved: {config_filepath}")

print("\n" + "="*60)
print("All results saved successfully!")
print(f"Output directory: {config.OUTPUT_PATH}")
print("="*60)

In [ ]:
# Cell 16: Summary Report

print("\n" + "="*80)
print("NARRATIVE SHIFT DETECTION - FINAL SUMMARY")
print("="*80)

print("\n📊 Dataset Statistics:")
print(f"  Topics processed: {len(config.TOPICS)}")
print(f"  Total windows: {len(all_windows)}")
for topic_name in config.TOPICS:
    print(f"    {topic_name:15s}: {len(topic_windows[topic_name]):6d} windows")

print("\n🧠 Model Configuration:")
print(f"  Architecture: Transformer-based TCL")
print(f"  Window size: {config.WINDOW_SIZE} days")
print(f"  Input dimension: {config.FINAL_DIM}")
print(f"  Hidden dimension: {config.HIDDEN_DIM}")
print(f"  Projection dimension: {config.PROJECTION_DIM}")
print(f"  Transformer layers: {config.NUM_LAYERS}")
print(f"  Attention heads: {config.NUM_HEADS}")

print("\n📈 Training:")
print(f"  Epochs: {config.EPOCHS}")
print(f"  Batch size: {config.BATCH_SIZE}")
print(f"  Final loss: {training_history['loss'][-1]:.4f}")

print("\n🔍 Shift Detection:")
total_shifts = sum(len(shifts) for shifts in topic_shifts.values())
print(f"  Total shifts detected: {total_shifts}")
for topic_name, shifts in topic_shifts.items():
    print(f"    {topic_name:15s}: {len(shifts):3d} shifts")

print("\n🎯 Pivot Sentences:")
total_pivots = sum(len(pivots) for pivots in pivot_results.values())
print(f"  Total pivot sentences found: {total_pivots}")
for topic_name, pivots in pivot_results.items():
    if len(pivots) > 0:
        print(f"    {topic_name:15s}: {len(pivots):2d} pivots")

print("\n💾 Outputs:")
print(f"  Model checkpoint: tcl_model_final.pt")
print(f"  Drift scores: drift_scores_[Topic].csv (x5)")
print(f"  Detected shifts: detected_shifts.json")
print(f"  Pivot sentences: pivot_sentences.json")
print(f"  Embeddings: topic_embeddings.pkl")
print(f"  Visualizations: drift_timeline_[Topic].png (x5)")
print(f"  Configuration: config.json")

print("\n" + "="*80)
print("✅ PIPELINE EXECUTION COMPLETE")
print("="*80)
print(f"\nAll results saved to: {config.OUTPUT_PATH}")
print("\nThank you for using the TCL Narrative Shift Detection Pipeline!")
print("="*80)